## Analyse des resultats

In [ ]:
import sys
sys.path.append('..')
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.visualization import *

## 1. Chargement des resultats de random Search

In [ ]:
with open('../runs/random_search_distilbert.json') as f:
    results_db = json.load(f)
with open('../runs/random_search_camembert.json') as f:
    results_cb = json.load(f)

df_db = pd.DataFrame([{'trial':r['trial_id'],'f1':r['f1'],'acc':r['accuracy'],'lr':r['params']['learning_rate'],'time_min':r['time_sec']/60} for r in results_db])
df_cb = pd.DataFrame([{'trial':r['trial_id'],'f1':r['f1'],'acc':r['accuracy'],'lr':r['params']['learning_rate'],'time_min':r['time_sec']/60} for r in results_cb])

print('=== DistilBERT ===')
display(df_db.sort_values('f1',ascending=False).round(4))
print('\n=== CamemBERT ===')
display(df_cb.sort_values('f1',ascending=False).round(4))

# 2.Graphique random search

In [ ]:
plot_random_search_comparison(results_db, results_cb, save_path='../figures/random_search.png')

## 3. Loss landscape

In [ ]:
data = np.load('../runs/loss_landscape.npz')
plot_loss_landscapes(
    data['alphas_db'], data['losses_db'],
    data['alphas_cb'], data['losses_cb'],
    save_path='../figures/loss_landscape.png'
)
print(f"Sharpness DistilBERT : {compute_sharpness(data['alphas_db'], data['losses_db']):.6f}")
print(f"Sharpness CamemBERT  : {compute_sharpness(data['alphas_cb'], data['losses_cb']):.6f}")

## 4.Courbe de convergence

In [ ]:
best_db = max(results_db, key=lambda x: x['f1'])
best_cb = max(results_cb, key=lambda x: x['f1'])
plot_convergence(best_db['convergence_history'], best_cb['convergence_history'],
                 save_path='../figures/convergence.png')

## 5.Analyse tokenizer

In [ ]:
import pandas as pd
df_tok = pd.read_csv('../runs/tokenizer_analysis.csv')
plot_tokenizer_analysis(df_tok, save_path='../figures/tokenizer_analysis.png')

## 6.Conclusion 

In [ ]:
best_f1_db = max(r['f1'] for r in results_db)
best_f1_cb = max(r['f1'] for r in results_cb)
delta = best_f1_cb - best_f1_db
print(f'Meilleur F1 DistilBERT : {best_f1_db:.4f}')
print(f'Meilleur F1 CamemBERT  : {best_f1_cb:.4f}')
print(f'Écart                  : {delta:+.4f}')
if delta > 0:
    print(f'→ CamemBERT surpasse DistilBERT de {delta:.4f} en F1')
else:
    print(f'→ DistilBERT résiste mieux que prévu (+{-delta:.4f})')